In [32]:
import findspark
findspark.init()
from pyspark.sql import SparkSession
from pyspark.sql.functions import *

# ============ 1. 创建Spark ============
spark = SparkSession.builder \
    .appName("SkewFullLearn") \
    .master("local[2]") \
    .getOrCreate()

# ============ 2. 制造倾斜数据 ============
data1 = [("1", f"data_{i}") for i in range(100000)]  # 倾斜key=1
data1 += [("2","a"), ("3","b"), ("4","c")]
df1 = spark.createDataFrame(data1, ["key", "value"])

data2 = [("1","北京"),("2","上海"),("3","深圳"),("4","广州")]
df2 = spark.createDataFrame(data2, ["key", "city"])
df1.show()
# df2.show()



ConnectionRefusedError: [WinError 10061] 由于目标计算机积极拒绝，无法连接。

In [ ]:
df2.show()

+---+----+
|key|city|
+---+----+
|  1|北京|
|  2|上海|
|  3|深圳|
|  4|广州|
+---+----+



In [ ]:
# ==============================================
# 方案1：广播join（解决join倾斜）
# ==============================================
result1 = df1.join(broadcast(df2), on="key")
# result1.show(50)
result1.sample(fraction=0.3).show()
result1.tail(10)
result1.orderBy(col('key').desc()).show(10)

+---+----------+----+
|key|     value|city|
+---+----------+----+
|  4|         c|广州|
|  3|         b|深圳|
|  2|         a|上海|
|  1|    data_0|北京|
|  1|data_50176|北京|
|  1|    data_1|北京|
|  1|data_50177|北京|
|  1|    data_2|北京|
|  1|data_50178|北京|
|  1|    data_3|北京|
+---+----------+----+
only showing top 10 rows



In [ ]:
# ==============================================
# 方案2：加盐打散 + 局部聚合 + 全局聚合（groupBy倾斜）
# ==============================================
# 加盐
df1_salt = df1.withColumn(
    "new_key", concat(col("key"), lit("_"), (rand() * 3).cast("int"))
)
df1_salt.show()

+---+-------+-------+
|key|  value|new_key|
+---+-------+-------+
|  1| data_0|    1_2|
|  1| data_1|    1_1|
|  1| data_2|    1_0|
|  1| data_3|    1_2|
|  1| data_4|    1_2|
|  1| data_5|    1_1|
|  1| data_6|    1_0|
|  1| data_7|    1_1|
|  1| data_8|    1_0|
|  1| data_9|    1_2|
|  1|data_10|    1_2|
|  1|data_11|    1_2|
|  1|data_12|    1_1|
|  1|data_13|    1_0|
|  1|data_14|    1_0|
|  1|data_15|    1_0|
|  1|data_16|    1_1|
|  1|data_17|    1_2|
|  1|data_18|    1_1|
|  1|data_19|    1_0|
+---+-------+-------+
only showing top 20 rows



In [ ]:
# 局部聚合
df_tmp = df1_salt.groupBy("new_key").count()
df_tmp.show()

+-------+-----+
|new_key|count|
+-------+-----+
|    1_0|33342|
|    1_1|33446|
|    1_2|33212|
|    4_1|    1|
|    2_2|    1|
|    3_2|    1|
+-------+-----+



In [ ]:
# 全局聚合
df_final = df_tmp.withColumn("key", split(col("new_key"), "_")[0]) \
                   .groupBy("key").sum("count")
df_final.show()

+---+----------+
|key|sum(count)|
+---+----------+
|  3|         1|
|  1|    100000|
|  4|         1|
|  2|         1|
+---+----------+



In [ ]:

# 必须先缓存并触发计算
df1.cache()
df1.count()

# 从 RDD 缓存里拿真实大小
for rdd_id, rdd in spark.sparkContext.getPersistentRDDs().items():
    size_bytes = rdd.memSize
    size_mb = size_bytes / 1024 / 1024
    size_gb = size_mb / 1024
    print(f"缓存分区: {rdd_id}")
    print(f"真实内存大小: {size_mb:.2f} MB ≈ {size_gb:.2f} GB")

AttributeError: 'SparkContext' object has no attribute 'getPersistentRDDs'

In [ ]:
spark.stop()

ConnectionRefusedError: [WinError 10061] 由于目标计算机积极拒绝，无法连接。